# From Manuscript Scan to Critical Edition: OCR, TEI Encoding, and Stemma Reconstruction

In digital humanities, textual criticism addresses an ancient problem: when a single work survives in several manuscripts that differ from one another, we want to know both which readings are closest to the original and which manuscripts copied from which. Computing has not made this simpler, but it has made it **reproducible and verifiable**: first OCR the manuscript scan into text, encode it in TEI standard as structured XML, then collate multiple **witnesses** character-by-character to extract differences into a **critical apparatus**, and finally use "shared errors" across manuscripts to infer their genealogical relationships and draw a **stemma codicum**.

The most critical and easily misunderstood step in this pipeline is stemma reconstruction. It rests on a simple but powerful principle—the **method of common errors**: if two manuscripts commit the same error in the same location, and that error is unlikely to have been invented independently by both, then they likely share a common ancestor. So textual criticism focuses not on "where they differ" but on "which differences were inherited together." The hard part is distinguishing between mere agreement and *meaningful* agreement: two manuscripts both transcribing a common word correctly tells us nothing, only shared **variants** carry genealogical information.

We use `socialverse` to walk through the complete pipeline: OCR→TEI→collation→stemma→final encoding. It is an analysis library for social sciences and humanities that registers each method in a function registry, validates at runtime whether prerequisites exist, and records every step in a reproducible provenance chain—you will see it at the end. In scope it parallels a suite of specialized digital humanities tools: `Tesseract` / `Kraken` for OCR, `TEI-P5` for encoding standards, `CollateX` / `Juxta` for multi-witness collation, and tools like `Stemmaweb` for stemma visualization.

Our corpus is a miniature "tradition": four witnesses A/B/C/D of the same sentence, with deliberate copying relationships embedded (B and C share a common spelling error), so the reconstructed stemma has a ground truth against which to check.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from matplotlib import font_manager

import socialverse as sv

# 让图里的中文标题正常渲染:挑一个装了的 CJK 字体(缺则回落,不报错)
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _cjk in ("Arial Unicode MS", "Songti SC", "STHeiti", "Hiragino Sans GB",
             "PingFang SC", "Noto Sans CJK SC", "Microsoft YaHei"):
    if _cjk in _installed:
        plt.rcParams["font.sans-serif"] = [_cjk, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False

## OCR and TEI Encoding: Making a Manuscript Page Computable

The first step in textual scholarship is entering a manuscript into machine-readable form, and that entry itself must follow a standard—otherwise each project transcribes differently and nothing is exchangeable. The standard for digital humanities is **TEI-P5**—an XML specification for describing texts, where `<teiHeader>` records metadata (title, responsibility, provenance) and `<body>` contains `<p>` for paragraphs and `<lb/>` for line breaks. `ocr_tei` combines both steps: it performs layout-aware OCR on images, then encodes the result as valid TEI.

This environment lacks Tesseract, so `ocr_tei` **degrades gracefully**—it treats a manually provided transcription as if it has already been OCR'd, and encodes it as valid TEI anyway. For teaching, this is precisely what needs emphasis: the function records in the provenance chain whether real OCR was used or the text was passed through directly, rather than pretending a scan occurred. Below we feed a transcription with archaic spelling, where `sources['scans']` uses a `{doc_id: page}` structure; each page can be an image path (which gets OCR'd) or, as here, a ready-made transcription.

In [2]:
st_tei = sv.StudyState()
st_tei.write("sources", "scans", {"folio1": "In the begynning was the Word."})

sv.pp.ocr_tei(st_tei, titles={"folio1": "Prologue, folio 1r"})  # titles 填进 teiHeader

# ocr_tei 把每一页各编码成一份 TEI,存成 {doc_id: TEI字符串} 的字典
tei_map = st_tei.corpus["tei"]
print("corpus['tei'] 的类型:", type(tei_map).__name__, "· 页面:", list(tei_map))
print("\n--- folio1 的 TEI-P5(前 12 行)---")
print("\n".join(tei_map["folio1"].splitlines()[:12]))

corpus['tei'] 的类型: dict · 页面: ['folio1']

--- folio1 的 TEI-P5(前 12 行)---
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
  <teiHeader>
    <fileDesc>
      <titleStmt><title>Prologue, folio 1r</title></titleStmt>
      <publicationStmt><p>Encoded by socialverse ocr-tei-encoding.</p></publicationStmt>
      <sourceDesc><p>OCR / text of source folio1.</p></sourceDesc>
    </fileDesc>
  </teiHeader>
  <text>
    <body>
        <p>In the begynning was the Word.</p>


The output is a valid TEI document: `<teiHeader>` includes the title we provided (`Prologue, folio 1r`), and `<body>` contains the transcribed text. Which OCR path was used is recorded in the provenance chain—here `engine` is `text-passthrough` and `ocr_available` is `False`, making it immediately clear that this page underwent no actual image recognition.

In [3]:
prov = st_tei.evidence["provenance"]  # 该槽存放最近一步的方法学记录
print("OCR 引擎     :", prov["engine"])
print("引擎可用     :", prov["ocr_available"])
print("识别语言     :", prov["lang"])
print("文档数       :", prov["n_documents"])

OCR 引擎     : text-passthrough
引擎可用     : False
识别语言     : eng
文档数       : 1


## Registering Multiple Witnesses

The raw material for collation is several copies of the same work. Before character-by-character collation, each witness must undergo two operations: **Unicode normalization** (reducing visually identical characters with different encodings to a single form like NFC, so alignment does not mistake them for variants), and segmentation into addressable units with **character offsets** (so the apparatus can point precisely to "character N through character M"). `build_corpus` does both at once and produces a `manifest` listing.

Our four witnesses are different transcriptions of the same sentence. We have **deliberately planted common errors**: B and C both miscopied `bright` as `bryght`—a spelling variant unlikely to be independently invented by two scribes, exactly the kind of signal the method of common errors seeks; C additionally changed `sea` to `ocean`; D stays closest to the base text A, adding only one extra word at the end. By the logic of common errors, we expect the reconstructed stemma to reveal that "B and C are related."

In [4]:
witnesses = {
    "A": "the moon was bright and the sea was calm that night",          # 底本(base)
    "B": "the moon was bryght and the sea was calm that nyght",          # 与 C 共享 bright→bryght
    "C": "the moon was bryght and the ocean was calm that night",        # 与 B 共享 bright→bryght;另有 sea→ocean
    "D": "the moon was bright and the sea was calm that night indeed",   # 最接近 A,仅句尾多一词
}

st = sv.StudyState()
st.write("sources", "corpora", witnesses)
sv.pp.build_corpus(st, unit="sentence")  # 按句切单元;也可 unit="word"/"line"

st.corpus["manifest"]  # 每个见证本切了几个单元、多少字符

,doc_id,n_units,n_chars
0,A,1,51
1,B,1,51
2,C,1,53
3,D,1,58


The `manifest` confirms the length differences: A and B each have 51 characters, C gains 2 characters due to `sea→ocean`, and D reaches 58 because of the extra word. Each unit carries its own id and character offset, which the apparatus will use to locate passages.

In [5]:
unit0 = st.corpus["units"][0]
print("unit_id  :", unit0["unit_id"])      # 形如 "A:0-51",见证本 + 字符区间
print("doc_id   :", unit0["doc_id"])
print("span     :", (unit0["start"], unit0["end"]))
print("text     :", unit0["text"])

unit_id  : A:0-51
doc_id   : A
span     : (0, 51)
text     : the moon was bright and the sea was calm that night


## Collation: Variants and Apparatus

This is the main act of collation. `philology_collate` aligns each witness character-by-character to base text A (using `difflib.SequenceMatcher`), sorts every difference into three types—**substitution (sub)**, **omission (om)**, **addition (add)**—and merges them by position into a **critical apparatus**. The apparatus is the machine-readable equivalent of the footnotes on a critical edition's page: each locus records the base text's lemma, each witness's variant at that location, and which witnesses agree with the base.

It depends on `corpus['documents']`, which `build_corpus` has already filled; we can run directly. We write `documents` again explicitly only to ensure the base text source matches `build_corpus` exactly.

In [6]:
st.write("corpus", "documents", witnesses)  # 与 build_corpus 用同一份文本作底本
sv.tl.philology_collate(st, base="A")        # 以 A 为底本对勘

apparatus = st.artifacts["apparatus"]
print(f"校勘记共 {len(apparatus)} 个 locus(异文点)")

# 排成一张编辑会印在书页脚的批判校勘表
app_rows = []
for e in apparatus:
    readings = "; ".join(f"{w}: {r}" for w, r in e["readings"].items())
    app_rows.append({
        "locus": e["locus"],
        "base_span": tuple(e["base_span"]),
        "lemma (底本读法)": e["lemma"],
        "异文 readings": readings,
        "与底本一致": ", ".join(e["agree_with_base"]) or "—",
    })
pd.DataFrame(app_rows)

校勘记共 4 个 locus(异文点)


,locus,base_span,lemma (底本读法),异文 readings,与底本一致
0,1,"(3, 4)",bright,B: bryght; C: bryght,D
1,2,"(6, 7)",sea,C: ocean,"B, D"
2,3,"(10, 11)",night,B: nyght,"C, D"
3,4,"(11, 11)",∅,D: indeed,"B, C"


Four loci capture all the differences in this small tradition: at locus 1 (`bright`), both B and C read `bryght` while D agrees with the base—this is our planted common error; locus 2 (`sea`) shows only C changed to `ocean`; locus 3 (`night`) shows only B wrote `nyght`; locus 4 is a word added only by D (lemma is empty `∅`, type is addition).

Looking at the complete structure of the first locus: `witness_types` marks each witness's edit type (substitution/omission/addition) at this site, and `agree_with_base` lists those witnesses matching the base—together, these let the stemma step compute "who diverged from the base in the same way."

In [7]:
e0 = apparatus[0]
print("locus          :", e0["locus"])
print("lemma (底本)   :", e0["lemma"])
print("readings       :", e0["readings"])         # 各见证本此处的异读
print("witness_types  :", e0["witness_types"])    # sub / om / add
print("与底本一致     :", e0["agree_with_base"])

locus          : 1
lemma (底本)   : bright
readings       : {'B': 'bryght', 'C': 'bryght'}
witness_types  : {'B': 'sub', 'C': 'sub'}
与底本一致     : ['D']


## Stemma Reconstruction

With the apparatus in hand, stemma reconstruction follows logically. `philology_collate` computes for each witness a "variant signature"—the pattern of how it diverges from the base across all loci—then computes **common error distance (Hamming)** between each pair: the more loci where two witnesses diverge from the base in the same way, the smaller their distance and the closer their kinship. Finally it uses `networkx`'s **minimum spanning tree** to connect these distances into a tree, yielding `models['stemma']`.

The results show A–D distance is smallest (1), because D differs only in one extra word at the end while matching the base elsewhere; A–B and A–C distances are both 2. B and C cluster together because they share `bright→bryght`, confirming the method of common errors' prediction.

In [8]:
stemma = st.models["stemma"]
print("root(底本)  :", stemma["root"])
print("method       :", stemma["method"])
print("\n谱系边(共同错误距离越小 = 亲缘越近):")
for edge in sorted(stemma["edges"], key=lambda x: x["shared_error_distance"]):
    print(f"  {edge['from']} — {edge['to']}   shared_error_distance = {edge['shared_error_distance']}")

root(底本)  : A
method       : minimum-spanning-tree over shared-error (Hamming) distance

谱系边(共同错误距离越小 = 亲缘越近):
  A — D   shared_error_distance = 1
  A — B   shared_error_distance = 2
  A — C   shared_error_distance = 2


Drawing the stemma makes it clearer. Edge weights are common error distances, smaller means closer; the red node is the nominal base text A.

In [9]:
g = nx.Graph()
for node in stemma["nodes"]:
    g.add_node(node)
for edge in stemma["edges"]:
    g.add_edge(edge["from"], edge["to"], weight=edge["shared_error_distance"])

pos = nx.spring_layout(g, seed=7, weight="weight")
fig, ax = plt.subplots(figsize=(6.2, 4.4))
root = stemma["root"]
node_colors = ["#c0392b" if n == root else "#2c3e50" for n in g.nodes()]
nx.draw_networkx_nodes(g, pos, node_color=node_colors, node_size=1500, ax=ax)
nx.draw_networkx_labels(g, pos, font_color="white", font_size=13, font_weight="bold", ax=ax)
nx.draw_networkx_edges(g, pos, width=2.0, edge_color="#7f8c8d", ax=ax)
edge_labels = {(u, v): d["weight"] for u, v, d in g.edges(data=True)}
nx.draw_networkx_edge_labels(g, pos, edge_labels=edge_labels, font_size=11, ax=ax)
ax.set_title(
    f"Stemma codicum — 谱系树(根/底本 = {root},红)\n边权 = 共同错误距离(Hamming)",
    fontsize=11,
)
ax.axis("off")
fig.tight_layout()
fig.savefig("fig_stemma.png", dpi=130, bbox_inches="tight")  # PNG 用 tight
plt.close(fig)
print("已保存谱系图 -> fig_stemma.png")

已保存谱系图 -> fig_stemma.png


![Stemma tree](fig_stemma.png)

The red node is base text A; B and C cluster together due to their shared error, D sits close to A—the reconstructed stemma correctly reproduces the copying relationships we embedded.

## Finalization: Encoding as Valid TEI-P5

Once collation is complete, the final deliverable is a **valid TEI-P5 XML document** that other projects can reference, search, and refine. `tei_encode` structures the base text's transcription into paragraphs `<p>` and line-break milestones `<lb/>`, adds a true `teiHeader` with title, author, and responsibility statement, and validates well-formedness with `lxml` (falling back to the standard library's `xml.etree` if `lxml` is unavailable).

In [10]:
sv.tl.tei_encode(
    st,
    witness="A",                                   # 把底本 A 编码为定稿
    title="On the Calm Sea",
    author="Anonymous (base witness A)",
    responsibility="socialverse tei-encoding demo",
)

tei_xml = st.corpus["tei"]  # 注意:这里是单份 XML 字符串,与第一步 ocr_tei 的逐页字典形态不同
print("--- 定稿 TEI-P5(teiHeader + body)---")
print(tei_xml)

with open("witness_A.tei.xml", "w", encoding="utf-8") as fh:
    fh.write(tei_xml)
print("已保存 -> witness_A.tei.xml")

--- 定稿 TEI-P5(teiHeader + body)---
<?xml version="1.0" encoding="UTF-8"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
  <teiHeader>
    <fileDesc>
      <titleStmt>
        <title>On the Calm Sea</title>
        <author>Anonymous (base witness A)</author>
        <respStmt>
          <resp>encoded by</resp>
          <name>socialverse tei-encoding demo</name>
        </respStmt>
      </titleStmt>
      <publicationStmt>
        <p>Encoded with socialverse (TEI P5).</p>
      </publicationStmt>
      <sourceDesc>
        <p>Born-digital transcription.</p>
      </sourceDesc>
    </fileDesc>
  </teiHeader>
  <text>
    <body>
        <p>the moon was bright and the sea was calm that night</p>
    </body>
  </text>
</TEI>

已保存 -> witness_A.tei.xml


Well-formedness is not a verbal promise but a validated fact recorded in the provenance chain: `well_formed=True`, which parser was used, and whether the root element is `TEI`, all left as traces.

In [11]:
val = st.evidence["provenance"]["validation"]  # tei_encode 最近写入的良构校验记录
print("良构       :", val["well_formed"])
print("解析器     :", val["parser"])
print("根元素     :", val["root"])
print("错误       :", val["error"])

良构       : True
解析器     : lxml
根元素     : TEI
错误       : None


## Reproducible Provenance Chain

Finally, see the key difference between `socialverse` and ad-hoc scripts. Over the entire pipeline, the study state automatically accumulates a **provenance ledger**: which function was used at each step, which slots it consumed, which it produced. This ledger is not manually maintained; it is automatically appended after each successful function call—a completed collation thus **brings its own audit trail**. In humanities and social sciences, "which step and which source produced this result" matters as much as the result itself.

In [12]:
print(st.summary())

print("\n--- provenance 账本(逐步的 requires→produces)---")
for rec in st.provenance:
    req = ", ".join(f"{s}[{','.join(k)}]" for s, k in rec["requires"].items()) or "∅"
    pro = ", ".join(f"{s}[{','.join(k)}]" for s, k in rec["produces"].items()) or "∅"
    fn = rec["function"].split(".")[-1]
    print(f"  step {rec['step']}: {fn}")
    print(f"           requires: {req}")
    print(f"           produces: {pro}")

StudyState
  sources: corpora
  corpus: documents, units, manifest, tei
  models: stemma
  evidence: provenance
  artifacts: apparatus, xml
  provenance: 3 step(s)

--- provenance 账本(逐步的 requires→produces)---
  step 1: build_corpus
           requires: sources[corpora]
           produces: corpus[documents,units,manifest], evidence[provenance]
  step 2: philology_collate
           requires: corpus[documents]
           produces: models[stemma], evidence[provenance], artifacts[apparatus]
  step 3: tei_encode
           requires: corpus[documents]
           produces: corpus[tei], artifacts[xml], evidence[provenance]


## Summary

We have completed a standard digital humanities collation pipeline: OCR → TEI encoding → registering witnesses → collating variants → producing apparatus → reconstructing stemma → final TEI. It parallels `Tesseract`/`Kraken` (OCR), `TEI-P5` (encoding standard), `CollateX`/`Juxta` (collation), and `Stemmaweb` (stemma), and it truly computes throughout: `difflib` for alignment, `networkx`'s minimum spanning tree for stemma.

Compared to that suite of ad-hoc tools, two things stand out here: first, functions **block you directly if prerequisites are missing** (calling `philology_collate` on an empty state raises `RegistryError` and tells you what to run first), rather than silently returning a wrong result; second, a provenance chain that spans the entire process and is automatically generated—exactly what ad-hoc tool chains lack and what textual scholarship most values. The next tutorial, [07_theory_lens_network](07_theory_lens_network.ipynb), shifts to a network view of social theory.